In [2]:
from pathlib import Path

from katabatic.artifacts import LocalArtifactStore
from katabatic.models.medgan.models import MEDGAN
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.utils.preprocess import preprocess_dataset

ROOT = None

for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "datasets").exists() and (p / "models").exists():
        ROOT = p
        break

raw_file = ROOT / "datasets" / "adult.csv"
processed_file = ROOT / "preprocessed_data" / "adult.csv"
artifact_dir = ROOT / "artifacts"

print("Raw dataset:", raw_file)
print("Dataset exists:", raw_file.exists())

processed_file.parent.mkdir(parents=True, exist_ok=True)

preprocess_dataset(
    str(raw_file),
    str(processed_file),
    target_col="class"
)

store = LocalArtifactStore(str(artifact_dir))

pipeline = TrainTestSplitPipeline(
    model=MEDGAN(
        ae_pretrain_epochs=10,
        gan_epochs=10
    )
)

pipeline._evaluations = []

results = pipeline.run(
    input_csv=str(processed_file),
    dataset_name="adult",
    artifact_store=store,
    model_name="medgan",
)

print(results)

Raw dataset: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\adult.csv
Dataset exists: True
Preprocessing: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\adult.csv
Saved preprocessed dataset to: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\preprocessed_data\adult.csv
Loaded data with shape: (32561, 15)


INFO:katabatic.models.medgan.models:================================================================================
INFO:katabatic.models.medgan.models:Training MedGAN Model
INFO:katabatic.models.medgan.models:================================================================================
INFO:katabatic.models.medgan.models:Loaded training data: (26048, 14)
INFO:katabatic.models.medgan.models:Categorical columns: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'class']
INFO:katabatic.models.medgan.models:Continuous columns: ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
INFO:katabatic.models.medgan.models:Data normalized to [0, 1] range
INFO:katabatic.models.medgan.models:Original range: [0.00, 1484705.00]
INFO:katabatic.models.medgan.models:Normalized range: [0.00, 1.00]


Train label distribution:
 class
<=50K    0.759175
>50K     0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
<=50K    0.759251
>50K     0.240749
Name: proportion, dtype: float64
Saved dataset artifact under datasets/adult/split-20260807-143853


INFO:katabatic.models.medgan.models:
Phase 1: Pretraining Autoencoder for 10 epochs...
INFO:katabatic.models.medgan.models:Epoch 1/10: AE Loss = 0.560280
INFO:katabatic.models.medgan.models:Epoch 10/10: AE Loss = 0.277658
INFO:katabatic.models.medgan.models:
Phase 2: Training GAN for 10 epochs...
INFO:katabatic.models.medgan.models:Epoch 1/10: D Loss = 1.137047, G Loss = 0.851512
INFO:katabatic.models.medgan.models:Epoch 10/10: D Loss = 0.110211, G Loss = 15.765408
INFO:katabatic.models.medgan.models:
Generating 26048 synthetic samples...
INFO:katabatic.models.medgan.models:
Synthetic data saved to: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\artifacts\models\medgan_adult_train-20260807-143853\synthetic
INFO:katabatic.models.medgan.models:Training complete!


{'message': 'Train test split pipeline executed successfully.', 'dataset_ref': DatasetRef(dataset_name='adult', dataset_version='split-20260807-143853'), 'model_ref': ModelRef(model_name='medgan', dataset_name='adult', dataset_version='split-20260807-143853', train_run_id='train-20260807-143853'), 'evaluation_refs': []}


In [3]:
import pandas as pd

from katabatic.pipeline.evaluation_pipeline import SyntheticEvaluationPipeline

splits_root = ROOT / "artifacts" / "datasets" / "adult"

split_dirs = sorted(
    [p for p in splits_root.glob("split-*") if p.is_dir()],
    key=lambda p: p.stat().st_mtime
)

latest_split = split_dirs[-1]

print("Using split:", latest_split)

train_df = pd.read_csv(
    latest_split / "train" / "train_full.csv"
)

test_df = pd.read_csv(
    latest_split / "test" / "test_full.csv"
)

target_col = "class"

categorical_cols = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
]

continuous_cols = [
    "age",
    "fnlwgt",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
]

model = pipeline.model

synthetic_df = model.sample(
    len(train_df),
    seed=42
)

print("Synthetic type:", type(synthetic_df))
print("\nSynthetic sample:")
print(synthetic_df.head())

evaluation_pipeline = SyntheticEvaluationPipeline(
    dimensions=[
        "fidelity",
        "utility",
        "diversity",
        "privacy",
        "consistency",
        "stability",
    ],
    categorical_cols=categorical_cols,
    continuous_cols=continuous_cols,
)

report = evaluation_pipeline.run(
    real_data=train_df,
    synthetic_data=synthetic_df,
    target_col=target_col,
    test_data=test_df,
    model=model,
)

print("\nComposite Score:", report.composite_score)

print("\nDimension Scores:")
for dimension, score in report.dimension_scores.items():
    print(f"{dimension}: {score}")

Using split: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\artifacts\datasets\adult\split-20260807-143853
Synthetic type: <class 'pandas.core.frame.DataFrame'>

Synthetic sample:
         age workclass        fnlwgt   education  education-num  \
0  42.844612         ?  263278.40625  Assoc-acdm       9.962110   
1  39.309727         ?  265535.21875  Assoc-acdm       9.801229   
2  40.641541         ?  262395.84375  Assoc-acdm       9.846057   
3  39.871441         ?  259154.40625  Assoc-acdm       9.829961   
4  40.996021         ?  259177.37500  Assoc-acdm       9.799324   

  marital-status occupation   relationship   race     sex  capital-gain  \
0       Divorced      Sales      Unmarried  White  Female   5444.303711   
1  Never-married      Sales  Not-in-family  White    Male   3991.613037   
2  Never-married      Sales  Not-in-family  White    Male   4357.726074   
3  Never-married      Sales  Not-in-family  White    Male   4132.369141   
4  Never